In [1]:
## 1. Setup & Imports

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression

In [3]:
## 2. Data Loading & Filtering

In [4]:
# Extract and load the dataset
df = pd.read_csv('cedint_f1_hourly_clean.csv.gz', compression='gzip')

# Convert 'timestamp_local' to datetime and set as the timeline index
df['timestamp_local'] = pd.to_datetime(df['timestamp_local'])
df = df.set_index('timestamp_local').sort_index()

# Filter the dataset to only keep the NW zone and HVAC system
df_nw_hvac = df[(df['zone'] == 'NW') & (df['load_type'] == 'HVAC')].copy()

C:\Users\joosv\AppData\Local\Temp\ipykernel_10556\3545114500.py:2: DtypeWarning: Columns (10: energy_source) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('cedint_f1_hourly_clean.csv.gz', compression='gzip')


In [5]:
## 3. Task D1: Chronological Train/Test Split

In [6]:
# Split the data chronologically based on the project requirements
train_data = df_nw_hvac['2020-06-01':'2022-06-30']
val_data = df_nw_hvac['2022-07-01':'2022-12-31']
test_data_2023 = df_nw_hvac['2023-01-01':'2023-12-31']
test_data_2025 = df_nw_hvac['2025-01-01':'2025-12-31']

# Print the shape of our splits
print(f"Total rows for NW HVAC: {len(df_nw_hvac)}")
print(f"-> Training data: {len(train_data)}")
print(f"-> Validation data: {len(val_data)}")
print(f"-> Test data (2023 + 2025): {len(test_data_2023) + len(test_data_2025)}")

Total rows for NW HVAC: 26830
-> Training data: 18240
-> Validation data: 4417
-> Test data (2023 + 2025): 2758


In [7]:
## 4. Task D1: Evaluation Metrics Framework

In [8]:
def evaluate_performance(y_true, y_pred):
    # Remove NaN values to prevent math errors during evaluation
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true_clean = y_true[mask]
    y_pred_clean = y_pred[mask]
    
    # 1. MAE
    mae = mean_absolute_error(y_true_clean, y_pred_clean)
    
    # 2. RMSE
    rmse = np.sqrt(mean_squared_error(y_true_clean, y_pred_clean))
    
    # 3. CV(RMSE) in percentage (ASHRAE Target: < 30%)
    mean_true = np.mean(y_true_clean)
    cv_rmse = (rmse / mean_true) * 100
    
    # 4. NMBE in percentage (ASHRAE Target: between -10% and +10%)
    nmbe = (np.sum(y_true_clean - y_pred_clean) / (mean_true * len(y_true_clean))) * 100
    
    # Print the results
    print("--- MODEL PERFORMANCE ---")
    print(f"MAE:      {mae:.2f} Wh")
    print(f"RMSE:     {rmse:.2f} Wh")
    print(f"CV(RMSE): {cv_rmse:.2f} %  (ASHRAE target: < 30%)")
    print(f"NMBE:     {nmbe:.2f} %  (ASHRAE target: -10% to +10%)")
    print("-------------------------")
    
    return mae, rmse, cv_rmse, nmbe

In [9]:
## 5. Task D2: Baseline 1 - Seasonal Naive

In [10]:
# Create a copy of the validation data
baseline_df = val_data.copy()

# Forecast is exactly the 'energy_wh' from 1 week (168 hours) ago
baseline_df['forecast_naive'] = baseline_df['energy_wh'].shift(168)

# Evaluate the Seasonal Naive model
print("Performance of the Seasonal Naive Baseline (NW HVAC):")
_ = evaluate_performance(baseline_df['energy_wh'], baseline_df['forecast_naive'])

Performance of the Seasonal Naive Baseline (NW HVAC):
--- MODEL PERFORMANCE ---
MAE:      48.87 Wh
RMSE:     102.25 Wh
CV(RMSE): 102.85 %  (ASHRAE target: < 30%)
NMBE:     -8.75 %  (ASHRAE target: -10% to +10%)
-------------------------


In [11]:
## 6. Task D2: Baseline 2 - Linear Regression

In [12]:
# Prepare the required features: hour, weekday, and temperature
df_lr = df_nw_hvac.copy()
df_lr['hour'] = df_lr.index.hour
df_lr['weekday'] = df_lr.index.weekday

# Drop rows with missing values (Linear Regression cannot handle NaNs)
df_lr_clean = df_lr.dropna(subset=['energy_wh', 'temp_c', 'hour', 'weekday'])

# Split back into chronological Train and Validate sets
train_lr = df_lr_clean['2020-06-01':'2022-06-30']
val_lr = df_lr_clean['2022-07-01':'2022-12-31']

# Define features (X) and target (y)
features = ['hour', 'weekday', 'temp_c']
target = 'energy_wh'

X_train = train_lr[features]
y_train = train_lr[target]
X_val = val_lr[features]
y_val = val_lr[target]

# Train (fit) the Linear Regression model on the training data
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict the energy consumption for the validation data
val_lr = val_lr.copy()
val_lr['forecast_lr'] = lr_model.predict(X_val)

# Evaluate the Linear Regression baseline
print("Performance of the Linear Regression Baseline (NW HVAC):")
_ = evaluate_performance(val_lr['energy_wh'], val_lr['forecast_lr'])

Performance of the Linear Regression Baseline (NW HVAC):
--- MODEL PERFORMANCE ---
MAE:      99.98 Wh
RMSE:     120.33 Wh
CV(RMSE): 113.68 %  (ASHRAE target: < 30%)
NMBE:     -2.02 %  (ASHRAE target: -10% to +10%)
-------------------------
